# How to use Instance Pools with Inference Components

---
#### This notebook demonstrates how to use instance pools (heterogeneous endpoint) witn Inference Components on SageMaker AI

**Scenario:**
- Create endpoint with instance pool of 5 different instance types and managed instance scaling enabled
- Deploy a model using Inference Components on the endpoint
- Increase number of IC copies to trigger instance scaling out
- Verify the scaling results and confirm IC placement
- Decrease number of IC copies to trigger instance scaling in
- Verify the results

---


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # clien

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating" or status == "Updating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating" or status == "Updating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Check Service Quotas for Instance Types

Verify in AWS Console that you have quota for instance type in your pool before creating the endpoint.

### Alternative: Check Quotas via AWS CLI

```
# List all SageMaker endpoint quotas
aws service-quotas list-service-quotas \
    --service-code sagemaker \
    --region YOUR_REGION \
    --query "Quotas[?contains(QuotaName, 'endpoint usage')].{Name:QuotaName,Value:Value}" \
    --output table

# Check specific instance type
aws service-quotas get-service-quota \
    --service-code sagemaker \
    --quota-code L-<quota-code> \
    --region YOUR_REGION
```

## Container setup

In [ ]:
model_id = "Qwen/Qwen3-4B" 
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

### We will creat two seperate environments for instance with 8 and 4 GPUs

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi20.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

# Note: If your model requires authentication, set HF_TOKEN as an environment variable
# or use AWS Secrets Manager. Never hardcode tokens in notebooks.
# Example: HF_TOKEN = os.environ.get('HF_TOKEN') or retrieve from Secrets Manager

tp4_env = {
    # "HF_TOKEN": os.environ.get('HF_TOKEN'),  # Uncomment if authentication needed
    "HF_MODEL_ID": model_id,
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_MAX_MODEL_LEN": "16384",
    "OPTION_TENSOR_PARALLEL_DEGREE": "4",
}

tp8_env = {
    # "HF_TOKEN": os.environ.get('HF_TOKEN'),  # Uncomment if authentication needed
    "HF_MODEL_ID": model_id,
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_MAX_MODEL_LEN": "16384",
    "OPTION_TENSOR_PARALLEL_DEGREE": "8",
}

## Inference Compoment Setup

### Heterogeneous Endpoint Configuration

**Key Features:**

**RoutingConfig**: Uses **LEAST_OUTSTANDING_REQUESTS (LOR)** routing strategy
- Routes each request to the instance with the fewest in-flight requests per model copy
- Higher-capacity instances process requests faster and drain queues more quickly
- They naturally receive proportionally more traffic without manual weight configuration
- Essential for heterogeneous endpoints where instance types differ in throughput capacity

**VariantInstanceProvisionTimeoutInSeconds**: Set to 3600 seconds (1 hour)
- Total window for procuring instances from the pool
- SageMaker retries on Insufficient Capacity errors within this window
- Moves to next priority pool once timeout expires
- Separate from model download and container startup timeouts

**Instance Pools**: Priority-based fallback (up to 5 pools)
- Priority 1 (highest): ml.g6e.12xlarge - Latest generation, best performance
- Priority 2-4: G6e and G6 variants - Good balance of performance and availability
- Priority 5 (lowest): ml.g5.12xlarge - Proven reliability, cost-effective fallback

### We define instance pool with 5 different instance types, enable Managed Instance Scaling and deploy 4 instances behind the endpoint

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InitialInstanceCount": 4,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "VariantInstanceProvisionTimeoutInSeconds": 900,
            "InstancePools": [
                {"InstanceType": "ml.g6.48xlarge", "Priority": 1},
                {"InstanceType": "ml.g6.24xlarge", "Priority": 2},
                {"InstanceType": "ml.g6.12xlarge", "Priority": 3},
                {"InstanceType": "ml.g5.48xlarge", "Priority": 4},
                {"InstanceType": "ml.g5.24xlarge", "Priority": 5},
            ],
            "ManagedInstanceScaling": {
                "Status": "ENABLED",
                "MinInstanceCount": 1,
                "MaxInstanceCount": 5,
            },
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        },
    ],
)

In [ ]:
_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

### Let's check the endpoint composition

### As you can see we have diverse set of instances behind out endpoint

In [ ]:
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
print(json.dumps(endpoint_info.pop("ProductionVariants", None), indent=2, default=str))

### Creating model object with different settings - we are going to create 2 model objects, one for instances with 8 GPUs (`g6.48xlarge` and `g5.48xlarge`) and another for instances with 4 GPUs

In [ ]:
tp8_model_name = f"tp8-{model_name}"

_ = sm.create_model(
    ModelName=tp8_model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": tp8_env,
    },
)

tp8_compute_req = {
    "MinMemoryRequiredInMb": 512,
    "NumberOfAcceleratorDevicesRequired": 8,
}

In [ ]:
tp4_model_name = f"tp4-{model_name}"

_ = sm.create_model(
    ModelName=tp4_model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": tp4_env,
    },
)

tp4_compute_req = {
    "MinMemoryRequiredInMb": 512,
    "NumberOfAcceleratorDevicesRequired": 4,
}


### In the next step, we are going to create one IC with different specification for each possible instance types in our instance pool. 

### Please note that although we have 4 instances in our endpoint, we are deploy just 3 IC => one instance will be idle

In [ ]:
ic_name = f"ic-{model_name}"

_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specifications=[
        {
            "InstanceType": "ml.g6.48xlarge",
            "ModelName": tp8_model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            },
            "ComputeResourceRequirements": tp8_compute_req,
        },
        {
            "InstanceType": "ml.g6.24xlarge",
            "ModelName": tp4_model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            },
            "ComputeResourceRequirements": tp4_compute_req,
        },
        {
            "InstanceType": "ml.g6.12xlarge",
            "ModelName": tp4_model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            },
            "ComputeResourceRequirements": tp4_compute_req,
        },
        {
            "InstanceType": "ml.g5.48xlarge",
            "ModelName": tp8_model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            },
            "ComputeResourceRequirements": tp8_compute_req,
        },
        {
            "InstanceType": "ml.g5.24xlarge",
            "ModelName": tp4_model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            },
            "ComputeResourceRequirements": tp4_compute_req,
        },
    ],
    RuntimeConfig={
        "CopyCount": 3,
    },
)
_ = wait_for_ic(ic_name)

### Let's inspect our IC placement

### We can see 3 IC, each one is placed on it's one instance

In [ ]:
ic_info = sm.describe_inference_component(InferenceComponentName=ic_name)
print(json.dumps(ic_info.pop("RuntimeConfig", None), indent=2, default=str))

### Inference test to make sure our IC deployed successfully

In [ ]:
payload={
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"]
print(f"-----------------------\n{usage}")

### Request 5 model copies to the endpoint (manual scaling out)

In [ ]:
_ = sm.update_inference_component_runtime_config(
    InferenceComponentName=ic_name,
    DesiredRuntimeConfig={
        'CopyCount': 5
    }
)

### We have create the endpoint with 4 instances, there are not enough instance to host 5 IC copies.

### This will trigger managed instance scaling to add 1 more instance to the endpoint and then add 2 more IC copies

***Please note `DesiredInstanceCount` below***

In [ ]:
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
print(json.dumps(endpoint_info.pop("ProductionVariants", None), indent=2, default=str))

### Let's check IC status

***Note `DesiredCopyCount` below***

In [ ]:
ic_info = sm.describe_inference_component(InferenceComponentName=ic_name)
print(json.dumps(ic_info.pop("RuntimeConfig", None), indent=2, default=str))

In [ ]:
_ = wait_for_endpoint(endpoint_name)

In [ ]:
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
print(json.dumps(endpoint_info.pop("ProductionVariants", None), indent=2, default=str))

### Endpoint has enough instances now, let's wait for additional IC to be ready

In [ ]:
_ = wait_for_ic(ic_name)

In [ ]:
ic_info = sm.describe_inference_component(InferenceComponentName=ic_name)
print(json.dumps(ic_info.pop("RuntimeConfig", None), indent=2, default=str))

### Now let's set number of IC to 1 and observe what happens

In [ ]:
_ = sm.update_inference_component_runtime_config(
    InferenceComponentName=ic_name,
    DesiredRuntimeConfig={
        'CopyCount': 1
    }
)

In [ ]:
ic_info = sm.describe_inference_component(InferenceComponentName=ic_name)
print(json.dumps(ic_info.pop("RuntimeConfig", None), indent=2, default=str))

In [ ]:
_ = wait_for_ic(ic_name)

### After about 5 minutes the number of instances was also scaled down to 1 (because of Managed Instance Scaling)

In [ ]:
time.sleep(600)
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
print(json.dumps(endpoint_info.pop("ProductionVariants", None), indent=2, default=str))

## Cleanup

In [ ]:
_ = sm.delete_inference_component(InferenceComponentName=ic_name)

In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

In [ ]:
_ = sm.delete_model(ModelName=tp8_model_name)
_ = sm.delete_model(ModelName=tp4_model_name)